# Notebook 2 — SQL Staging & Cleaning (SQLite)

**Goal (this notebook):** Load the raw CSV into SQLite (`raw_loans`), then create a typed, model-ready table (`clean_loans`) with consistent formats and engineered features.

---

## Outputs
- `data/db/credit_risk.db`
  - `raw_loans` (raw ingest)
  - `clean_loans` (typed + engineered)

## Key transformations in `clean_loans`
- `term` → `term_months` (INTEGER)
- `emp_length` → `emp_length_yrs` (INTEGER)
- `issue_d`, `earliest_cr_line`, `last_pymnt_d`, `last_credit_pull_d` → ISO month date text (`YYYY-MM-01`)
- `fico_avg` (REAL)


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

## 1) Paths & database connection
We keep the DB inside the repo under `data/db/`.


In [2]:
DATA_PATH = Path("data/raw/accepted_2007_to_2018Q4.csv")
assert DATA_PATH.exists(), f"File not found: {DATA_PATH.resolve()}"

DB_DIR = Path("data/db")
DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "credit_risk.db"
DB_PATH

PosixPath('data/db/credit_risk.db')

In [3]:
con = sqlite3.connect(DB_PATH)

con.execute("PRAGMA journal_mode = WAL;")
con.execute("PRAGMA synchronous = NORMAL;")
con.execute("PRAGMA temp_store = MEMORY;")
con.execute("PRAGMA cache_size = -200000;") 

## 2) Create `raw_loans` 

This is a production-style load: stream the CSV in chunks and append into SQLite.

In [4]:
# Chunked load settings
CHUNK_SIZE = 100_000  # adjust if needed
total_rows = 0

for i, chunk in enumerate(pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False)):
    chunk.columns = [c.strip() for c in chunk.columns]
    chunk.to_sql("raw_loans", con, if_exists="append", index=False)
    total_rows += len(chunk)
    if (i + 1) % 5 == 0:
        print(f"Loaded {total_rows:,} rows...")

print(f"Done. Total loaded rows: {total_rows:,}")

Loaded 500,000 rows...
Loaded 1,000,000 rows...
Loaded 1,500,000 rows...
Loaded 2,000,000 rows...
Done. Total loaded rows: 2,260,701


In [5]:
con.execute("SELECT COUNT(*) FROM raw_loans;").fetchone() #row count

(2260701,)

In [6]:
# Peek schema (first 12 columns)
con.execute("PRAGMA table_info(raw_loans);").fetchall()[:12]

[(0, 'id', 'INTEGER', 0, None, 0),
 (1, 'member_id', 'REAL', 0, None, 0),
 (2, 'loan_amnt', 'REAL', 0, None, 0),
 (3, 'funded_amnt', 'REAL', 0, None, 0),
 (4, 'funded_amnt_inv', 'REAL', 0, None, 0),
 (5, 'term', 'TEXT', 0, None, 0),
 (6, 'int_rate', 'REAL', 0, None, 0),
 (7, 'installment', 'REAL', 0, None, 0),
 (8, 'grade', 'TEXT', 0, None, 0),
 (9, 'sub_grade', 'TEXT', 0, None, 0),
 (10, 'emp_title', 'TEXT', 0, None, 0),
 (11, 'emp_length', 'TEXT', 0, None, 0)]

## 3) Build `clean_loans`
Create a clean, typed table using SQL casts + rule-based transforms.

### Month-abbreviation 
The LendingClub month strings look like `Dec-2018`. We convert them into ISO month dates (`2018-12-01`).

In [7]:
MONTH_CASE_TEMPLATE = (
    "CASE substr(trim({col}), 1, 3) "
    "WHEN 'Jan' THEN '01' "
    "WHEN 'Feb' THEN '02' "
    "WHEN 'Mar' THEN '03' "
    "WHEN 'Apr' THEN '04' "
    "WHEN 'May' THEN '05' "
    "WHEN 'Jun' THEN '06' "
    "WHEN 'Jul' THEN '07' "
    "WHEN 'Aug' THEN '08' "
    "WHEN 'Sep' THEN '09' "
    "WHEN 'Oct' THEN '10' "
    "WHEN 'Nov' THEN '11' "
    "WHEN 'Dec' THEN '12' "
    "ELSE NULL END"
)

def iso_month_date_sql(colname: str) -> str:
    month_case = MONTH_CASE_TEMPLATE.format(col=colname)
    return f"""CASE
  WHEN {colname} IS NULL THEN NULL
  WHEN length(trim({colname})) < 7 THEN NULL
  ELSE
    printf('%s-%s-01',
      substr(trim({colname}), 5, 4),
      {month_case}
    )
END"""

# Example (prints first 200 chars)
print(iso_month_date_sql("issue_d")[:200], "...")


CASE
  WHEN issue_d IS NULL THEN NULL
  WHEN length(trim(issue_d)) < 7 THEN NULL
  ELSE
    printf('%s-%s-01',
      substr(trim(issue_d), 5, 4),
      CASE substr(trim(issue_d), 1, 3) WHEN 'Jan' THEN ...


In [8]:
con.execute("DROP TABLE IF EXISTS clean_loans;")
con.commit()

In [9]:
create_clean_sql = f'''
CREATE TABLE clean_loans AS
SELECT
  -- identifiers
  rowid AS row_id,
  CAST(id AS TEXT) AS id,

  -- dates (stored as ISO month-date text)
  {iso_month_date_sql("issue_d")}              AS issue_date,
  {iso_month_date_sql("earliest_cr_line")}     AS earliest_cr_line_date,
  {iso_month_date_sql("last_pymnt_d")}         AS last_pymnt_date,
  {iso_month_date_sql("last_credit_pull_d")}   AS last_credit_pull_date,

  -- geography / product
  CAST(addr_state AS TEXT) AS addr_state,
  CAST(purpose AS TEXT) AS purpose,
  CAST(grade AS TEXT) AS grade,
  CAST(sub_grade AS TEXT) AS sub_grade,
  CAST(initial_list_status AS TEXT) AS initial_list_status,

  -- loan terms
  CAST(loan_amnt AS REAL) AS loan_amnt,
  CAST(funded_amnt AS REAL) AS funded_amnt,
  CAST(funded_amnt_inv AS REAL) AS funded_amnt_inv,

  -- your sample shows these as numeric strings already (no % sign)
  CAST(int_rate AS REAL) AS int_rate_pct,
  CAST(installment AS REAL) AS installment,

  -- term: " 36 months" -> 36
  CAST(TRIM(REPLACE(term, 'months', '')) AS INTEGER) AS term_months,

  -- employment length mapping
  CASE
    WHEN emp_length IS NULL THEN NULL
    WHEN emp_length = 'n/a' THEN NULL
    WHEN emp_length LIKE '10+%' THEN 10
    WHEN emp_length LIKE '< 1%' THEN 0
    ELSE CAST(REPLACE(REPLACE(emp_length, ' years', ''), ' year', '') AS INTEGER)
  END AS emp_length_yrs,

  CAST(home_ownership AS TEXT) AS home_ownership,
  CAST(annual_inc AS REAL) AS annual_inc,
  CAST(verification_status AS TEXT) AS verification_status,
  CAST(dti AS REAL) AS dti,

  -- credit history / behaviour
  CAST(delinq_2yrs AS INTEGER) AS delinq_2yrs,
  CAST(inq_last_6mths AS INTEGER) AS inq_last_6mths,
  CAST(open_acc AS INTEGER) AS open_acc,
  CAST(pub_rec AS INTEGER) AS pub_rec,
  CAST(revol_bal AS REAL) AS revol_bal,
  CAST(revol_util AS REAL) AS revol_util_pct,
  CAST(total_acc AS INTEGER) AS total_acc,

  CAST(fico_range_low AS REAL) AS fico_range_low,
  CAST(fico_range_high AS REAL) AS fico_range_high,
  (CAST(fico_range_low AS REAL) + CAST(fico_range_high AS REAL)) / 2.0 AS fico_avg,

  -- portfolio accounting (useful for realised outcomes / KPIs)
  CAST(out_prncp AS REAL) AS out_prncp,
  CAST(total_pymnt AS REAL) AS total_pymnt,
  CAST(total_rec_prncp AS REAL) AS total_rec_prncp,
  CAST(total_rec_int AS REAL) AS total_rec_int,
  CAST(total_rec_late_fee AS REAL) AS total_rec_late_fee,
  CAST(recoveries AS REAL) AS recoveries,
  CAST(collection_recovery_fee AS REAL) AS collection_recovery_fee,
  CAST(last_pymnt_amnt AS REAL) AS last_pymnt_amnt,

  -- target/outcome
  CAST(loan_status AS TEXT) AS loan_status

FROM raw_loans;
'''
con.execute(create_clean_sql)
con.commit()
print("clean_loans created.")


clean_loans created.


## 4) Validation queries
Checks to confirm transformations worked.


In [10]:
# Row counts should match
raw_n = con.execute("SELECT COUNT(*) FROM raw_loans;").fetchone()[0]
clean_n = con.execute("SELECT COUNT(*) FROM clean_loans;").fetchone()[0]
raw_n, clean_n

(2260701, 2260701)

In [11]:
# term_months distribution
pd.read_sql_query(
    "SELECT term_months, COUNT(*) AS n FROM clean_loans GROUP BY term_months ORDER BY term_months;",
    con
)

,term_months,n
0,NaN,33
1,36.0,1609754
2,60.0,650914


In [12]:
# emp_length mapping sanity check
pd.read_sql_query(
    '''
    SELECT emp_length_yrs, COUNT(*) AS n
    FROM clean_loans
    GROUP BY emp_length_yrs
    ORDER BY emp_length_yrs;
    ''',
    con
).head(15)

,emp_length_yrs,n
0,NaN,146940
1,0.0,189988
2,1.0,148403
3,2.0,203677
4,3.0,180753
5,4.0,136605
6,5.0,139698
7,6.0,102628
8,7.0,92695
9,8.0,91914


In [ ]:
# Date parsing: check min/max
pd.read_sql_query(
    '''
    SELECT
      MIN(issue_date) AS issue_min,
      MAX(issue_date) AS issue_max,
      MIN(earliest_cr_line_date) AS ecl_min,
      MAX(earliest_cr_line_date) AS ecl_max,
      MIN(last_pymnt_date) AS last_pymnt_min,
      MAX(last_pymnt_date) AS last_pymnt_max
    FROM clean_loans;
    ''',
    con
)


In [13]:
# loan_status distribution (top)
pd.read_sql_query(
    '''
    SELECT loan_status, COUNT(*) AS n
    FROM clean_loans
    GROUP BY loan_status
    ORDER BY n DESC
    LIMIT 20;
    ''',
    con
)

,loan_status,n
0,Fully Paid,1076751
1,Current,878317
2,Charged Off,268559
3,Late (31-120 days),21467
4,In Grace Period,8436
5,Late (16-30 days),4349
6,Does not meet the credit policy. Status:Fully ...,1988
7,Does not meet the credit policy. Status:Charge...,761
8,Default,40
9,None,33


## 5) Add basic indexes
Indexes help a lot once you start KPI queries (grade, issue_date, loan_status, etc.).

In [14]:
con.execute("CREATE INDEX IF NOT EXISTS idx_clean_issue_date ON clean_loans(issue_date);")
con.execute("CREATE INDEX IF NOT EXISTS idx_clean_grade ON clean_loans(grade);")
con.execute("CREATE INDEX IF NOT EXISTS idx_clean_sub_grade ON clean_loans(sub_grade);")
con.execute("CREATE INDEX IF NOT EXISTS idx_clean_loan_status ON clean_loans(loan_status);")
con.commit()

"Indexes created."

'Indexes created.'